# XGBoost Model
**ShipmentSure — Predicting On-Time Delivery**

- XGBClassifier for boosted-tree modeling
- GridSearchCV for hyperparameter tuning
- Full evaluation: confusion matrix, feature importance, ROC curve

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import xgboost as xgb
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve, classification_report
)

sns.set_theme(style="whitegrid")
print("Libraries loaded.")

ModuleNotFoundError: No module named 'xgboost'

In [3]:
# Load processed data (or fallback to raw)
try:
    X_train = pd.read_csv('../data/processed/X_train.csv')
    X_test = pd.read_csv('../data/processed/X_test.csv')
    y_train = pd.read_csv('../data/processed/y_train.csv').values.ravel()
    y_test = pd.read_csv('../data/processed/y_test.csv').values.ravel()
    print("Loaded processed data.")
except FileNotFoundError:
    from sklearn.pipeline import Pipeline
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler, OneHotEncoder

    df = pd.read_csv('../data/Train.csv')
    if 'ID' in df.columns:
        df = df.drop('ID', axis=1)
    X = df.drop('Reached.on.Time_Y.N', axis=1)
    y = df['Reached.on.Time_Y.N'].values

    num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
    cat_cols = X.select_dtypes(include='object').columns.tolist()

    preprocessor = ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                          ('enc', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols)
    ])
    X_raw_train, X_raw_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    X_train = preprocessor.fit_transform(X_raw_train)
    X_test = preprocessor.transform(X_raw_test)
    print("Generated processed data from raw.")

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Loaded processed data.
Train: (800, 19), Test: (200, 19)


In [ ]:
# XGBoost with hyperparameter tuning
print("Tuning XGBoost...")
xgb_model = xgb.XGBClassifier(eval_metric='logloss', random_state=42, use_label_encoder=False)
param_grid_xgb = {
    'n_estimators': [50, 100],
    'learning_rate': [0.1, 0.2],
    'max_depth': [3, 5]
}
grid_xgb = GridSearchCV(xgb_model, param_grid_xgb, cv=5, scoring='f1')
grid_xgb.fit(X_train, y_train)
xgb_best = grid_xgb.best_estimator_
print(f"Best params: {grid_xgb.best_params_}")

In [ ]:
# Evaluate on test set
y_pred = xgb_best.predict(X_test)
y_prob = xgb_best.predict_proba(X_test)[:, 1]

metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred, zero_division=0),
    'Recall': recall_score(y_test, y_pred, zero_division=0),
    'F1-Score': f1_score(y_test, y_pred, zero_division=0),
    'ROC-AUC': roc_auc_score(y_test, y_prob)
}

print("XGBoost Evaluation:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

In [4]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('XGBoost Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

NameError: name 'confusion_matrix' is not defined

In [5]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"XGBoost (AUC={metrics['ROC-AUC']:.3f})")
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('XGBoost ROC Curve')
plt.legend()
plt.tight_layout()
plt.show()

NameError: name 'roc_curve' is not defined

In [6]:
# Feature Importance
fi = xgb_best.feature_importances_
fi_df = pd.DataFrame({'Feature': [f'f{i}' for i in range(len(fi))], 'Importance': fi})
fi_df = fi_df[fi_df['Importance'] > 0].sort_values('Importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=fi_df.sort_values('Importance'), palette='Blues_d')
plt.title('XGBoost Feature Importances (Top 15)')
plt.tight_layout()
plt.show()

NameError: name 'xgb_best' is not defined

In [ ]:
# Classification report
print("Classification Report — XGBoost:")
print(classification_report(y_test, y_pred))